In [1]:
import sys
sys.path.insert(0, 'src')
import importlib
import run_pathway_materials
importlib.reload(run_pathway_materials)
from run_pathway_materials import run_pathway_materials
import pandas as pd
import plotly.express as px
import os
import math
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import re
from pathlib import Path
import shared.utils as _shared_utils

from Plot_functions import def_elec_positive, def_priv_mob_positive ,plot_mult_positive, plot_new_positive, plot_all_material_demand

# Construction automatique du fichier excel et .dat

In [2]:
from run_build_mi import main
main()                        # équivalent aux valeurs par défaut du CLI
#main(scenario='optimiste')    # applique les overrides du scénario "optimiste"
#main(write_dat=False)         # ne régénère que le`` xlsx

[build_table] computed 690 tech intensities in 3.2s
[build_table] kept 0 existing rows for techs outside the Mapping sheet in 71.2s
[build_table] built 183540 rows for the Mapping sheet's 690 technologies in 74.4s
[build_table] wrote technologies_mi_all_years.xlsx (183540 rows, 42560 vehicle-detail rows) in 145.4s
[build_table] wrote Material_intensity.dat in 161.8s
[build_table] total: 161.8s

Coverage report:
  not_mapped       : 498
  not_yet_modeled  : 0
  placeholder_zero : 0
  integrated       : 192

not_mapped (498):
  - ALKALINE_ELECTROLYSIS
  - AL_MAKING
  - AL_MAKING_HR
  - AN_DIG
  - AN_DIG_SI
  - ATR
  - ATR_CCS
  - BATTERY
  - BIOETHANOL_TO_BIOJETFUELS
  - BIOGAS_ATR
  - BIOGAS_ATR_CCS
  - BIOGAS_BIOMETHANE
  - BIOGAS_SMR
  - BIOGAS_SMR_CCS
  - BIOMASS_GAS_EF_H2
  - BIOMASS_GAS_EF_H2_CCS
  - BIOMASS_GAS_FB_H2
  - BIOMASS_GAS_FB_H2_CCS
  - BIOMETHANE_TO_BIOMETHANOL
  - BIOMETHANOL_CARBONYLATION
  - BIOMETHANOL_FT
  - BIOMETHANOL_TO_AROMATICS
  - BIOMETHANOL_TO_OLEFINS
  - B

# Run Pathway Materials sans aucune contrainte -> Resultat Pathway

In [2]:

results_materials = run_pathway_materials('my_first_run_materials', verbose=False, skip_if_exists=True)

#results_materials['F_new']                    # capacités nouvelles [Phases, Technologies]
#results_materials['Material_content_year']    # demande matériau annualisée [Years, Technologies, Materials]
#results_materials['Recycled_material']

[run_pathway_materials] my_first_run_materials — pkl exists, loading from disk.


In [3]:
material_content = (results_materials['Material_content_year']['Material_content_year']
                     .groupby(['Technologies', 'Materials']).sum() * 5)

In [4]:
elec_techs_positive = def_elec_positive(results_materials)
priv_mob_techs_positive = def_priv_mob_positive(results_materials)


In [5]:
plot_new_positive(results_materials, elec_techs_positive)

In [6]:
plot_new_positive(results_materials, priv_mob_techs_positive, sector='priv_mob')

In [7]:
plot_mult_positive(results_materials, sector= 'elec_prod')

In [8]:
plot_mult_positive(results_materials,sector='priv_mob')

In [9]:
plot_all_material_demand(results_materials, sector = 'priv_mob')

# Run Pathway Materials sans contrainte minium de wind, hydro et pv (plan hydro quebec)

In [14]:
results_materials_relax = run_pathway_materials('my_first_run_materials_relax', verbose=True, hydro_quebec_constraints =False)


Presolve eliminates 0 constraints and 113710 variables.
Adjusted problem:
709736 variables:
	6342 binary variables
	84 nonlinear variables
	703310 linear variables
772300 constraints; 3632798 nonzeros
	7 nonlinear constraints
	772293 linear constraints
	632221 equality constraints
	136636 inequality constraints
	3443 range constraints
2 objectives, all linear; 2 nonzeros.

Gurobi 13.0.2:   pre:dual = -1
  alg:method = 2
  bar:crossover = 0
  tech:threads = 0
  pre:passes = 3
  bar:convtol = 9.9999999999999995e-07
  pre:solve = -1
  iis:find = 1
Set parameter LogToConsole to value 1
  tech:outlev = 1

AMPL MP initial flat model has 709736 variables (0 integer, 6342 binary);
Objectives: 1 linear; 
Constraints:  772300 linear;

AMPL MP final model has 713179 variables (0 integer, 6342 binary);
Objectives: 1 linear; 
Constraints:  766925 linear;


Set parameter InfUnbdInfo to value 1
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[x86] - Darwin 23.5.0 23F79)

CPU model: Intel(R) C

In [15]:
elec_positive_relax = def_elec_positive(results_materials_relax)

In [16]:
plot_elec_new_positive(results_materials_relax, elec_positive_relax)

In [17]:
plot_elec_mult_positive(results_materials_relax, elec_positive_relax)

In [ ]:
plot_all_material_demand(results_materials_relax)

NameError: name 'results_materials_relax' is not defined

# Run Pathway Materials sans contrainte minium de wind, hydro et pv (plan hydro quebec) et avec limite matériau

In [17]:
results_materials_relax_limit = run_pathway_materials('my_first_run_materials_relax_limit', verbose=True, hydro_quebec_constraints =False, materials_limit= True)


Presolve eliminates 0 constraints and 113710 variables.
Adjusted problem:
709736 variables:
	6342 binary variables
	84 nonlinear variables
	703310 linear variables
772300 constraints; 3632756 nonzeros
	7 nonlinear constraints
	772293 linear constraints
	632221 equality constraints
	136636 inequality constraints
	3443 range constraints
2 objectives, all linear; 2 nonzeros.

Gurobi 13.0.2:   pre:dual = -1
  alg:method = 2
  bar:crossover = 0
  tech:threads = 0
  pre:passes = 3
  bar:convtol = 9.9999999999999995e-07
  pre:solve = -1
  iis:find = 1
Set parameter LogToConsole to value 1
  tech:outlev = 1

AMPL MP initial flat model has 709736 variables (0 integer, 6342 binary);
Objectives: 1 linear; 
Constraints:  772300 linear;

AMPL MP final model has 713179 variables (0 integer, 6342 binary);
Objectives: 1 linear; 
Constraints:  766927 linear;


Set parameter InfUnbdInfo to value 1
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[x86] - Darwin 23.5.0 23F79)

CPU model: Intel(R) C

In [18]:
elec_positive_relax_limit = def_elec_positive(results_materials_relax_limit)

In [19]:
plot_elec_new_positive(results_materials_relax_limit, elec_positive_relax_limit)

In [20]:
plot_elec_mult_positive(results_materials_relax_limit, elec_positive_relax_limit)

In [21]:
plot_all_material_demand(results_materials_relax_limit)